<a href="https://colab.research.google.com/github/kellllyyy/mase/blob/main/tutorial_2_lora_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 2: Finetuning Bert for Sequence Classification using a LoRA adapter

When we import a pretrained transformer model from HuggingFace, we receive the encoder/decoder weights, which aren't that useful on their own - to perform a useful task such as sequence classification, we add a classification head on top of the model and train those weights on the required dataset. In this tutorial, we'll look at fine tuning a Bert model for sequence classification with two approaches. First, we'll attempt full Supervised Fine Tuning (SFT). Then, we'll use the Mase stack to add a [LoRA](https://arxiv.org/abs/2106.09685) adapter to the model. We'll look at the effect in memory requirement for training and the achieved accuracy.

In [1]:
%pip uninstall -y optimum transformers datasets evaluate fsspec
%pip install optimum==1.24.0 transformers==4.51 datasets==3.3.2 evaluate==0.4.3 fsspec==2024.12.0

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: fsspec 2025.3.0
Uninstalling fsspec-2025.3.0:
  Successfully uninstalled fsspec-2025.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled token

In [2]:
%pip install \
black toml GitPython colorlog \
cocotb==1.9.2 pybind11 \
timm pytorch-nlp sentencepiece einops kornia \
hyperopt@git+https://github.com/hyperopt/hyperopt.git \
optuna stable-baselines3[extra] \
tensorboard tensorboardx wandb lightning pytorch-lightning \
bitarray "bitstring>=4.2" attr-dot-dict emoji \
onnx onnxruntime onnxconverter-common accelerate \
absl-py prettytable pynvml py-cpuinfo psutil cvxpy pydot \
h5py imageio imageio-ffmpeg \
opencv-python matplotlib scikit-learn pandas \
sphinx-rtd-theme "sphinx-needs>=4" \
sphinx-test-reports@git+https://github.com/useblocks/sphinx-test-reports \
sphinxcontrib-plantuml sphinx-glpi-theme \
ghp-import myst_parser myst-nb sphinx-book-theme \
ultralytics==8.3.235


  Cloning https://github.com/hyperopt/hyperopt.git to /tmp/pip-install-b7vvf2v3/hyperopt_e87f1a73e9d54213aa1c3ad2f6cd9b5b
  Running command git clone --filter=blob:none --quiet https://github.com/hyperopt/hyperopt.git /tmp/pip-install-b7vvf2v3/hyperopt_e87f1a73e9d54213aa1c3ad2f6cd9b5b
  Resolved https://github.com/hyperopt/hyperopt.git to commit 0658f680c84a313eaffe1771a20dfe2ebabd7fd4
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/useblocks/sphinx-test-reports to /tmp/pip-install-b7vvf2v3/sphinx-test-reports_058892d4b0bf43f99f5547eb847f510f
  Running command git clone --filter=blob:none --quiet https://github.com/useblocks/sphinx-test-reports /tmp/pip-install-b7vvf2v3/sphinx-test-reports_058892d4b0bf43f99f5547eb847f510f
  Resolved https://github.com/useblocks/sphinx-test-reports to commit 97df9daec4a50800ecb5cfdcd5a49416840750bf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/mase/src')
print(f"Added '/content/drive/MyDrive/mase/src' to sys.path")

Added '/content/drive/MyDrive/mase/src' to sys.path


In [3]:
checkpoint = "DeepWokLab/bert-tiny"
tokenizer_checkpoint = "DeepWokLab/bert-tiny"
dataset_name = "imdb"

## Sentiment Analysis with the IMDb Dataset

The IMDB dataset, introduced in [this 2011 paper](https://aclanthology.org/P11-1015/) from Stanford, is commonly used for sentiment analysis in the Natural Language Processing (NLP) community. This is a collection of 50k movie reviews from the IMDb website, labelled as either "positive" or "negative". Here is an example of a positive review:

> I turned over to this film in the middle of the night and very nearly skipped right passed it. It was only because there was nothing else on that I decided to watch it. In the end, I thought it was great.<br /><br />An interesting storyline, good characters, a clever script and brilliant directing makes this a fine film to sit down and watch. This was, in fact, the first I'd heard of this movie, but I would have been happy to have paid money to see this at the cinema.<br /><br />My IMDB Rating : 8 out of 10<br /><br />

And a negative review:

> its a totally average film with a few semi-alright action sequences that make the plot seem a little better and remind the viewer of the classic van dam films. parts of the plot don't make sense and seem to be added in to use up time. the end plot is that of a very basic type that doesn't leave the viewer guessing and any twists are obvious from the beginning. the end scene with the flask backs don't make sense as they are added in and seem to have little relevance to the history of van dam's character. not really worth watching again, bit disappointed in the end production, even though it is apparent it was shot on a low budget certain shots and sections in the film are of poor directed quality

The dataset is available from HuggingFace through the ``datasets`` library. We use the `get_tokenized_dataset` utility in Mase to automatically tokenize it.

In [4]:
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

## Generate a MaseGraph with Custom Arguments

By inspecting the implementation of the Bert model in HuggingFace, we can see the forward function has a signature similar to the following.

```python
    class BertForSequenceClassification(BertPreTrainedModel):
        def __init__(self, config):
            super().__init__(config)
            self.bert = BertModel(config)
            ...

        def forward(
            self,
            input_ids: Optional[torch.Tensor] = None,
            attention_mask: Optional[torch.Tensor] = None,
            token_type_ids: Optional[torch.Tensor] = None,
            position_ids: Optional[torch.Tensor] = None,
            head_mask: Optional[torch.Tensor] = None,
            inputs_embeds: Optional[torch.Tensor] = None,
            labels: Optional[torch.Tensor] = None,
            output_attentions: Optional[bool] = None,
            output_hidden_states: Optional[bool] = None,
            return_dict: Optional[bool] = None,
        ) -> Union[Tuple[torch.Tensor], SequenceClassifierOutput]:
            ...
```

By default, the MaseGraph constructor chooses to use the `input_ids` argument, ignoring the other optional arguments. However, you can specify which inputs to drive during symbolic tracing using the `hf_input_names` argument. In the following cell, we also drive the `attention_mask` and `labels` inputs. By specifying the `labels` argument, we include a `nn.CrossEntropyLoss` module at the end of the model to calculate the loss directly.

> **Task:** Remove the `attention_mask` and `labels` arguments from the `hf_input_names` list and re-run the following cell. Use `mg.draw()` to visualize the graph in each case. Can you see any changes in the graph topology? Can you explain why this happens?

In [5]:
from transformers import AutoModelForSequenceClassification

from chop import MaseGraph
import chop.passes as passes

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg)

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepWokLab/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.
W0202 14:23:03.883000 911 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
INFO     Getting dummy input for DeepWokLab/bert-tiny.


tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]],


        [[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]]])
tensor([[[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       

## Full Supervised Finetuning (SFT)

Before training the model, let's inspect how many trainable parameters there are. If you're familiar with Keras, you might have used the `model.summary()` API before, but it's not as easy to do the same in Pytorch - luckily, Mase has a module-level pass with this functionality.

In [6]:
from chop.passes.module import report_trainable_parameters_analysis_pass

_, _ = report_trainable_parameters_analysis_pass(mg.model)

+-------------------------------------------------+------------------------+
| Submodule                                       |   Trainable Parameters |
+=================================================+========================+
| bert                                            |                4385920 |
+-------------------------------------------------+------------------------+
| bert.embeddings                                 |                3972864 |
+-------------------------------------------------+------------------------+
| bert.embeddings.word_embeddings                 |                3906816 |
+-------------------------------------------------+------------------------+
| bert.embeddings.token_type_embeddings           |                    256 |
+-------------------------------------------------+------------------------+
| bert.embeddings.position_embeddings             |                  65536 |
+-------------------------------------------------+------------------------+

From this, we can see the majority of the trainable parameters are in the `Embedding` layer. We don't need to train this, so we freeze those parameters in the cell below.

In [7]:
for param in mg.model.bert.embeddings.parameters():
    param.requires_grad = False

To train the model, we rely on the `Trainer` class from the `transformers` library, which makes it easy to set up a training loop with any hardware configuration. The `get_trainer` utility in Mase handles assigning the training arguments to the `Trainer` class for common use cases, such as in this tutorial.

In [8]:
from chop.tools import get_trainer

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

/content/drive/MyDrive/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Before running any fine tuning, let's see how the model performs out of the box. Without any fine-tuning, we can see the model just performs a random guess - there are two labels in the dataset, so this corresponds to an accuracy of around 50%.

In [9]:
# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

Evaluation accuracy: 0.50896


Now, run the cell below to execute a single training epoch with the current setup.

In [10]:
trainer.train()

Step,Training Loss
500,0.630700
1000,0.519000
1500,0.477800
2000,0.432700
2500,0.429600
3000,0.419400


TrainOutput(global_step=3125, training_loss=0.48224215087890626, metrics={'train_runtime': 51.6016, 'train_samples_per_second': 484.481, 'train_steps_per_second': 60.56, 'total_flos': 0.0, 'train_loss': 0.48224215087890626, 'epoch': 1.0})

Let's see how much accuracy we get after a single training epoch of full finetuning.

In [11]:
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

Evaluation accuracy: 0.81644


We can now export the SFT version of the model to be used in later tutorials.

In [12]:
from pathlib import Path

# mg.export(f"{Path.home()}/tutorial_2_sft")

mg.export("/content/drive/MyDrive/mase/src/tutorial_2_sft")

INFO     Exporting MaseGraph to /content/drive/MyDrive/mase/src/tutorial_2_sft.pt, /content/drive/MyDrive/mase/src/tutorial_2_sft.mz
INFO     Exporting GraphModule to /content/drive/MyDrive/mase/src/tutorial_2_sft.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /content/drive/MyDrive/mase/src/tutorial_2_sft.mz
WARNING  Failed to pickle call_function node: finfo
WARNING  cannot pickle 'torch.finfo' object
WARNING  Failed to pickle call_function node: getattr_2
WARNING  cannot pickle 'torch.finfo' object


## Parameter Efficient Finetuning (PEFT) with LoRA

An alternative to full fine-tuning is Parameter Efficient Fine Tuning (PEFT), which uses a small number of trainable parameters to achieve similar performance. LoRA was proposed by a research team at Microsoft in 2021, as an efficient technique for PEFT.

<div style="text-align: center;">
    <img src="imgs/lora_adapter.png" alt="drawing" width="400"/>
</div>

Consider the standard equation of a linear layer:

$$
y = X W + b
$$

The LoRA method involves replacing this with the following, where A and B are low-rank matrices. We freeze the $W$ parameters, and only allow the optimizer to train the parameters in $A$ and $B$.

$$
y = X (W + AB) + b
$$

This enables us to achieve accuracies comparable to full fine tuning, while only training a fraction of the parameters. See [the paper](https://arxiv.org/abs/2106.09685) for more details. We can inject the LoRA adapter into the existing model using the `insert_lora_adapter_transform_pass` pass in Mase, as follows.

In [13]:
mg, _ = passes.insert_lora_adapter_transform_pass(
    mg,
    pass_args={
        "rank": 6,
        "alpha": 1.0,
        "dropout": 0.5,
    },
)

INFO     Replaced node: bert_encoder_layer_0_attention_self_query, target: bert.encoder.layer.0.attention.self.query with LoRALinear module.
INFO     Replaced node: bert_encoder_layer_0_attention_self_key, target: bert.encoder.layer.0.attention.self.key with LoRALinear module.
INFO     Replaced node: bert_encoder_layer_0_attention_self_value, target: bert.encoder.layer.0.attention.self.value with LoRALinear module.
INFO     Replaced node: bert_encoder_layer_0_attention_output_dense, target: bert.encoder.layer.0.attention.output.dense with LoRALinear module.
INFO     Replaced node: bert_encoder_layer_0_intermediate_dense, target: bert.encoder.layer.0.intermediate.dense with LoRALinear module.
INFO     Replaced node: bert_encoder_layer_0_output_dense, target: bert.encoder.layer.0.output.dense with LoRALinear module.
INFO     Replaced node: bert_encoder_layer_1_attention_self_query, target: bert.encoder.layer.1.attention.self.query with LoRALinear module.
INFO     Replaced node: bert_enco

Similar to before, let's report the number of trainable parameters.

In [14]:
_, _ = report_trainable_parameters_analysis_pass(mg.model)

+-----------------------------------------------------+------------------------+
| Submodule                                           |   Trainable Parameters |
+=====================================================+========================+
| bert                                                |                 439808 |
+-----------------------------------------------------+------------------------+
| bert.embeddings                                     |                      0 |
+-----------------------------------------------------+------------------------+
| bert.embeddings.word_embeddings                     |                      0 |
+-----------------------------------------------------+------------------------+
| bert.embeddings.token_type_embeddings               |                      0 |
+-----------------------------------------------------+------------------------+
| bert.embeddings.position_embeddings                 |                      0 |
+---------------------------

In this case, LoRA reduces the number of trainable parameters by $4.5\times$! We'll run a few more training epochs and evaluate the resulting accuracy.

In [15]:
trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=1,
)
trainer.train()

# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

/content/drive/MyDrive/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.433300
1000,0.417600
1500,0.419200
2000,0.390200
2500,0.398300
3000,0.390800


Evaluation accuracy: 0.83496


After training is finished, we can run the `fuse_lora_weights_transform_pass` pass to optimize the model for inference. This pass replaces each `LoRALinear` instance with an `nn.Linear` module, where the $AB$ product added to the original weights matrix. This incurs less kernel invocations when deploying the model, which reduces inference runtime.

In [16]:
mg, _ = passes.fuse_lora_weights_transform_pass(mg)
eval_results = trainer.evaluate()

INFO     Fusing LoRALinear weights for bert.encoder.layer.0.attention.self.query.
INFO     Fusing LoRALinear weights for bert.encoder.layer.0.attention.self.key.
INFO     Fusing LoRALinear weights for bert.encoder.layer.0.attention.self.value.
INFO     Fusing LoRALinear weights for bert.encoder.layer.0.attention.output.dense.
INFO     Fusing LoRALinear weights for bert.encoder.layer.0.intermediate.dense.
INFO     Fusing LoRALinear weights for bert.encoder.layer.0.output.dense.
INFO     Fusing LoRALinear weights for bert.encoder.layer.1.attention.self.query.
INFO     Fusing LoRALinear weights for bert.encoder.layer.1.attention.self.key.
INFO     Fusing LoRALinear weights for bert.encoder.layer.1.attention.self.value.
INFO     Fusing LoRALinear weights for bert.encoder.layer.1.attention.output.dense.
INFO     Fusing LoRALinear weights for bert.encoder.layer.1.intermediate.dense.
INFO     Fusing LoRALinear weights for bert.encoder.layer.1.output.dense.
INFO     Fusing LoRALinear weights f

In [17]:
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

Evaluation accuracy: 0.83496


Finally, export the finetuned model to be used in future tutorials.

In [18]:
from pathlib import Path

# mg.export(f"{Path.home()}/tutorial_2_lora")
mg.export("/content/drive/MyDrive/mase/src/tutorial_2_lora")

INFO     Exporting MaseGraph to /content/drive/MyDrive/mase/src/tutorial_2_lora.pt, /content/drive/MyDrive/mase/src/tutorial_2_lora.mz
INFO     Exporting GraphModule to /content/drive/MyDrive/mase/src/tutorial_2_lora.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /content/drive/MyDrive/mase/src/tutorial_2_lora.mz
WARNING  Failed to pickle call_function node: finfo
WARNING  cannot pickle 'torch.finfo' object
WARNING  Failed to pickle call_function node: getattr_2
WARNING  cannot pickle 'torch.finfo' object


## Conclusion

By adjusting the rank number of LoRA, we can control the trade-off between memory usage and fine-tuned accuracy. Such parameter-efficient fine-tuning techniques are very useful in the area of large language models (LLMs), where the memory requirement for training is a significant bottleneck.